# Medewerkerroutes Heerlen — 20 medewerkers, 5 cliënten per medewerker

Dit notebook berekent de optimale dagroutes voor **20 medewerkers**:
- **Start**: thuisadres medewerker
- **5 stops**: optimaal gekozen cliënten uit `clients.csv` (100 beschikbaar)
- **Einde**: terug naar huis

Aanpak:
1. Laad het wegennet en bouw een NetworkX-graph.
2. Laad medewerkers (thuis) en cliënten — koppel beiden aan het dichtstbijzijnde netwerkknooppunt.
3. Wijs 5 unieke cliënten toe aan elke medewerker (nearest-neighbor greedy, zodat elke cliënt max. 1× bezocht wordt).
4. Optimaliseer de volgorde van de 5 stops per medewerker (nearest-neighbor TSP: thuis → stop1 → … → stop5 → thuis).
5. Teken de routes op een interactieve Folium-kaart met 20 unieke kleuren.

## 1. Bibliotheken importeren

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from shapely import wkt
import folium
from scipy.spatial import cKDTree
from itertools import permutations
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


## 2. Wegennet laden en graph bouwen

In [2]:
edges_df = pd.read_csv('../output/heerlen_edge_table.csv')
print(f'Edges loaded: {len(edges_df)}')
edges_df['geometry'] = edges_df['geometry'].apply(wkt.loads)

G = nx.Graph()
node_coords = {}  # node_id -> (lon, lat)

for _, row in edges_df.iterrows():
    geom = row['geometry']
    coords = list(geom.coords)
    u, v = row['u'], row['v']
    G.add_edge(u, v, weight=row['travel_time_min'], geometry=geom)
    node_coords[u] = (coords[0][0],  coords[0][1])
    node_coords[v] = (coords[-1][0], coords[-1][1])

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.')

# k-d tree for fast nearest-node lookup
node_ids      = list(node_coords.keys())
node_lons_arr = np.array([node_coords[n][0] for n in node_ids])
node_lats_arr = np.array([node_coords[n][1] for n in node_ids])
tree = cKDTree(np.column_stack((node_lons_arr, node_lats_arr)))

# Edge geometry lookup (both directions)
edge_geom = {}
for _, row in edges_df.iterrows():
    edge_geom[(row['u'], row['v'])] = row['geometry']
    edge_geom[(row['v'], row['u'])] = row['geometry']

Edges loaded: 7183
Graph: 3120 nodes, 4340 edges.


## 3. Medewerkers laden

In [3]:
# ── Helper: snap a (lon, lat) pair to the nearest graph node ──────────────
def nearest_node(lon, lat):
    _, idx = tree.query([lon, lat])
    return node_ids[idx]

# ── Employee coordinates (manually geocoded from Heerlen addresses) ────────
# Source: employees.csv  (addresses are all in Heerlen, NL)
employee_data = [
    ('employees 1',  50.8872, 5.9812),
    ('employees 2',  50.8895, 5.9820),
    ('employees 3',  50.8883, 5.9830),
    ('employees 4',  50.8855, 5.9795),
    ('employees 5',  50.8945, 5.9660),
    ('employees 6',  50.8878, 5.9808),
    ('employees 7',  50.8948, 5.9700),
    ('employees 8',  50.8870, 5.9825),
    ('employees 9',  50.8868, 5.9817),
    ('employees 10', 50.8785, 5.9750),
    ('employees 11', 50.8840, 5.9810),
    ('employees 12', 50.8860, 5.9835),
    ('employees 13', 50.8850, 5.9880),
    ('employees 14', 50.8890, 5.9822),
    ('employees 15', 50.8710, 5.9920),
    ('employees 16', 50.8810, 5.9680),
    ('employees 17', 50.8952, 5.9672),
    ('employees 18', 50.8875, 5.9805),
    ('employees 19', 50.8940, 5.9665),
    ('employees 20', 50.8790, 5.9760),
]
employees_df = pd.DataFrame(employee_data, columns=['name', 'lat', 'lon'])
employees_df['home_node'] = employees_df.apply(
    lambda r: nearest_node(r['lon'], r['lat']), axis=1
)
print(f'Employees loaded: {len(employees_df)}')
print(employees_df[['name', 'lat', 'lon', 'home_node']].to_string())

Employees loaded: 20
            name      lat     lon   home_node
0    employees 1  50.8872  5.9812    42031708
1    employees 2  50.8895  5.9820    42035060
2    employees 3  50.8883  5.9830    42035060
3    employees 4  50.8855  5.9795    42030147
4    employees 5  50.8945  5.9660    42043919
5    employees 6  50.8878  5.9808  3345862100
6    employees 7  50.8948  5.9700  5113236587
7    employees 8  50.8870  5.9825    42031708
8    employees 9  50.8868  5.9817    42031708
9   employees 10  50.8785  5.9750    42020609
10  employees 11  50.8840  5.9810    42027869
11  employees 12  50.8860  5.9835    42032567
12  employees 13  50.8850  5.9880  1724265714
13  employees 14  50.8890  5.9822    42035060
14  employees 15  50.8710  5.9920    42007239
15  employees 16  50.8810  5.9680    42022056
16  employees 17  50.8952  5.9672    42044957
17  employees 18  50.8875  5.9805  3345862100
18  employees 19  50.8940  5.9665    42038645
19  employees 20  50.8790  5.9760    42020890


## 4. Cliënten laden uit clients.csv

In [4]:
clients_df = pd.read_csv('../output/clients.csv')
print('Original columns:', clients_df.columns.tolist())
print('First 3 rows:\n', clients_df.head(3))

# Auto-detect coordinate column (string containing two numbers)
coord_col = None
for col in clients_df.columns:
    sample = clients_df[col].dropna().astype(str).iloc[0]
    parts = sample.replace(',', ' ').replace(';', ' ').split()
    if len(parts) == 2:
        try:
            float(parts[0]); float(parts[1])
            coord_col = col
            break
        except ValueError:
            pass

if coord_col is None:
    coord_col = clients_df.columns[0]
    print(f'No coordinate column detected, using first column: {coord_col}')
else:
    print(f'Using coordinate column: "{coord_col}"')

def split_coords(s):
    parts = str(s).replace(';', ' ').replace(',', ' ').split()
    if len(parts) == 2:
        return float(parts[0]), float(parts[1])
    return np.nan, np.nan

clients_df[['lat', 'lon']] = clients_df[coord_col].apply(
    lambda x: pd.Series(split_coords(x))
)
clients_df = clients_df.dropna(subset=['lat', 'lon']).reset_index(drop=True)
clients_df['client_id'] = clients_df.index  # stable integer ID
clients_df['node'] = clients_df.apply(
    lambda r: nearest_node(r['lon'], r['lat']), axis=1
)

print(f'Clients loaded: {len(clients_df)}')
print(clients_df[['client_id', 'lat', 'lon', 'node']].head())

Original columns: ['name', 'address', 'coordinates', 'care_arrangement', 'preferences', 'time_window_start', 'time_window_end', 'care_hours', 'dogs', 'cats', 'smokes']
First 3 rows:
        name                              address           coordinates  \
0  Client 1  Kloosterkoolhof 26D, 6415XT Heerlen   50.8910272 5.987952   
1  Client 2   Frans Halsstraat 6, 6415TH Heerlen  50.9003376 5.9904336   
2  Client 3         Hertstraat 1, 6414CH Heerlen  50.9155146 5.9745851   

  care_arrangement preferences time_window_start time_window_end  care_hours  \
0         HBH Plus     morning             08:00           12:00         2.0   
1         HBH Plus   afternoon             12:00           18:00         2.0   
2   Wash & Ironing     morning             08:00           12:00         2.5   

   dogs  cats  smokes  
0     2     0   False  
1     1     1   False  
2     0     0   False  
Using coordinate column: "coordinates"
Clients loaded: 100
   client_id        lat       lon        nod

## 5. Reistijdenmatrix berekenen

We berekenen de kortste reistijd (Dijkstra) vanuit elk thuisknooppunt én elk cliëntknooppunt naar alle andere knooppunten.  
Dit geeft ons alle benodigde pairwise reistijden.

In [5]:
# Collect all unique nodes we need shortest paths FROM
all_source_nodes = set(employees_df['home_node'].tolist() + clients_df['node'].tolist())
print(f'Computing Dijkstra from {len(all_source_nodes)} unique source nodes...')

# dist_lookup[src][dst] = travel time in minutes
dist_lookup = {}
for i, src in enumerate(all_source_nodes):
    dist_lookup[src] = nx.single_source_dijkstra_path_length(G, src, weight='weight')
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(all_source_nodes)} done')

def travel_time(node_a, node_b):
    """Return travel time (min) between two graph nodes."""
    return dist_lookup.get(node_a, {}).get(node_b, float('inf'))

print('Distance lookup ready.')

Computing Dijkstra from 102 unique source nodes...
  10/102 done
  20/102 done
  30/102 done
  40/102 done
  50/102 done
  60/102 done
  70/102 done
  80/102 done
  90/102 done
  100/102 done
Distance lookup ready.


## 6. Cliënttoewijzing — greedy nearest-neighbor

Elke cliënt mag maximaal **één keer** bezocht worden.  
We lopen per medewerker in volgorde en kennen steeds de 5 dichtstbijzijnde nog-beschikbare cliënten toe.

In [6]:
N_STOPS = 5  # clients per employee

available = set(clients_df['client_id'].tolist())  # pool of unassigned clients
assignments = {}  # emp_idx -> list of client_ids

for emp_idx, emp in employees_df.iterrows():
    home_node = emp['home_node']
    # Score all available clients by travel time from home
    scored = [
        (travel_time(home_node, clients_df.loc[cid, 'node']), cid)
        for cid in available
    ]
    scored.sort()
    chosen = [cid for _, cid in scored[:N_STOPS]]
    assignments[emp_idx] = chosen
    available -= set(chosen)
    print(f'{emp["name"]}: clients {chosen}')

print(f'\nClients remaining unassigned: {len(available)}')

employees 1: clients [32, 36, 76, 21, 7]
employees 2: clients [73, 0, 82, 42, 35]
employees 3: clients [68, 31, 83, 64, 61]
employees 4: clients [38, 78, 25, 59, 37]
employees 5: clients [14, 63, 93, 16, 43]
employees 6: clients [28, 74, 13, 50, 12]
employees 7: clients [90, 58, 97, 45, 87]
employees 8: clients [72, 17, 89, 85, 39]
employees 9: clients [71, 24, 22, 15, 65]
employees 10: clients [23, 92, 62, 69, 91]
employees 11: clients [51, 94, 95, 47, 29]
employees 12: clients [34, 9, 60, 18, 66]
employees 13: clients [81, 49, 88, 6, 3]
employees 14: clients [54, 46, 1, 77, 56]
employees 15: clients [19, 99, 55, 30, 79]
employees 16: clients [27, 96, 48, 86, 70]
employees 17: clients [41, 80, 2, 10, 11]
employees 18: clients [98, 4, 53, 26, 5]
employees 19: clients [52, 8, 44, 67, 33]
employees 20: clients [57, 40, 84, 20, 75]

Clients remaining unassigned: 0


## 7. Routevolgorde optimaliseren per medewerker (nearest-neighbor TSP)

Voor elke medewerker optimaliseren we de volgorde van de 5 stops met een **nearest-neighbor heuristiek**:  
Start thuis → bezoek steeds de dichtstbijzijnde nog-niet-bezochte stop → keer terug naar huis.

In [7]:
def nn_tour(home_node, client_nodes):
    """
    Nearest-neighbor TSP heuristic.
    Returns ordered list of nodes: [home, stop1, stop2, ..., stopN, home]
    and the total travel time.
    """
    unvisited = list(client_nodes)
    tour = [home_node]
    current = home_node
    total_time = 0.0

    while unvisited:
        # Find nearest unvisited node
        best_time = float('inf')
        best_node = None
        for node in unvisited:
            t = travel_time(current, node)
            if t < best_time:
                best_time = t
                best_node = node
        tour.append(best_node)
        total_time += best_time
        unvisited.remove(best_node)
        current = best_node

    # Return home
    total_time += travel_time(current, home_node)
    tour.append(home_node)
    return tour, total_time


# Build tour for each employee
tours = {}  # emp_idx -> {'tour_nodes': [...], 'total_time': float, 'client_nodes': [...]}

for emp_idx, emp in employees_df.iterrows():
    home_node = emp['home_node']
    chosen_ids = assignments[emp_idx]
    chosen_nodes = [clients_df.loc[cid, 'node'] for cid in chosen_ids]

    tour_nodes, total_time = nn_tour(home_node, chosen_nodes)
    tours[emp_idx] = {
        'tour_nodes':   tour_nodes,
        'client_nodes': chosen_nodes,
        'client_ids':   chosen_ids,
        'total_time':   total_time,
    }
    print(f'{emp["name"]}: {total_time:.1f} min total, stops={len(chosen_nodes)}')

employees 1: 1.1 min total, stops=5
employees 2: 3.0 min total, stops=5
employees 3: 7.8 min total, stops=5
employees 4: 1.9 min total, stops=5
employees 5: 7.8 min total, stops=5
employees 6: 10.7 min total, stops=5
employees 7: 10.4 min total, stops=5
employees 8: 9.2 min total, stops=5
employees 9: 12.0 min total, stops=5
employees 10: 9.4 min total, stops=5
employees 11: 14.9 min total, stops=5
employees 12: 10.6 min total, stops=5
employees 13: 16.1 min total, stops=5
employees 14: 20.1 min total, stops=5
employees 15: 10.5 min total, stops=5
employees 16: 15.5 min total, stops=5
employees 17: 10.9 min total, stops=5
employees 18: 24.4 min total, stops=5
employees 19: 19.5 min total, stops=5
employees 20: 18.3 min total, stops=5


## 8. Kaart bouwen

Elke medewerker krijgt een unieke kleur. De route volgt de echte wegen uit het wegennet.

In [8]:
# 20 visually distinct colors
EMPLOYEE_COLORS = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45', '#fabed4',
    '#469990', '#dcbeff', '#9A6324', '#ff8c00', '#800000',
    '#aaffc3', '#808000', '#00bfff', '#000075', '#a9a9a9',
]

def nodes_to_latlon(path_nodes):
    """
    Convert a list of graph node IDs to a list of (lat, lon) coordinates
    by walking each consecutive edge and reading its full geometry.
    This correctly handles edges that are curved or multi-point linestrings.
    """
    latlon = []
    for i in range(len(path_nodes) - 1):
        u, v = path_nodes[i], path_nodes[i + 1]
        geom = edge_geom.get((u, v))
        if geom is not None:
            coords = list(geom.coords)  # list of (lon, lat)
            # Reverse if the geometry runs from v to u
            u_coord = node_coords.get(u)
            if u_coord and len(coords) > 0:
                if abs(coords[0][0] - u_coord[0]) > abs(coords[-1][0] - u_coord[0]):
                    coords = coords[::-1]
            latlon.extend([(lat, lon) for lon, lat in coords])
        else:
            # Straight-line fallback (should rarely happen)
            cu, cv = node_coords.get(u), node_coords.get(v)
            if cu:
                latlon.append((cu[1], cu[0]))
            if cv:
                latlon.append((cv[1], cv[0]))
    return latlon


def get_road_latlon(node_a, node_b):
    """
    Compute the shortest path between two nodes and return its
    road-following (lat, lon) coordinates.
    """
    try:
        path = nx.shortest_path(G, source=node_a, target=node_b, weight='weight')
        return nodes_to_latlon(path)
    except nx.NetworkXNoPath:
        ca, cb = node_coords.get(node_a), node_coords.get(node_b)
        result = []
        if ca:
            result.append((ca[1], ca[0]))
        if cb:
            result.append((cb[1], cb[0]))
        return result


# Map center
center_lat = np.mean(node_lats_arr)
center_lon = np.mean(node_lons_arr)

m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')

# ── Road network background ──────────────────────────────────────────────
for _, row in edges_df.iterrows():
    latlon = [(lat, lon) for lon, lat in row['geometry'].coords]
    folium.PolyLine(locations=latlon, color='#cccccc', weight=1, opacity=0.35).add_to(m)

# ── Employee routes ──────────────────────────────────────────────────────
stop_labels = ['thuis', 'stop 1', 'stop 2', 'stop 3', 'stop 4', 'stop 5', 'thuis']

for emp_idx, emp in employees_df.iterrows():
    color   = EMPLOYEE_COLORS[emp_idx % len(EMPLOYEE_COLORS)]
    t_nodes = tours[emp_idx]['tour_nodes']  # [home, s1, s2, s3, s4, s5, home]

    for seg_i in range(len(t_nodes) - 1):
        latlon = get_road_latlon(t_nodes[seg_i], t_nodes[seg_i + 1])
        if len(latlon) >= 2:
            label = f"{emp['name']} | {stop_labels[seg_i]} → {stop_labels[seg_i + 1]}"
            folium.PolyLine(
                locations=latlon,
                color=color,
                weight=4,
                opacity=0.85,
                tooltip=label
            ).add_to(m)

# ── Home markers ────────────────────────────────────────────────────────
for emp_idx, emp in employees_df.iterrows():
    color = EMPLOYEE_COLORS[emp_idx % len(EMPLOYEE_COLORS)]
    tour  = tours[emp_idx]
    folium.Marker(
        location=[emp['lat'], emp['lon']],
        icon=folium.DivIcon(
            html=(
                f'<div style="width:20px;height:20px;background:{color};'
                'border:3px solid white;border-radius:50%;'
                'box-shadow:0 1px 5px rgba(0,0,0,.5);"></div>'
            ),
            icon_size=(20, 20),
            icon_anchor=(10, 10)
        ),
        popup=folium.Popup(
            f"<b>{emp['name']}</b><br>"
            f"Totale reistijd: {tour['total_time']:.1f} min<br>"
            f"Cliënten: {tour['client_ids']}",
            max_width=220
        ),
        tooltip=f"{emp['name']} (thuis)"
    ).add_to(m)

# ── Client stop markers ───────────────────────────────────────────────────
client_color_map = {}
for emp_idx in range(len(employees_df)):
    color = EMPLOYEE_COLORS[emp_idx % len(EMPLOYEE_COLORS)]
    emp_name = employees_df.loc[emp_idx, 'name']
    for stop_num, cid in enumerate(tours[emp_idx]['client_ids'], start=1):
        client_color_map[cid] = (color, emp_name, stop_num)

for _, client in clients_df.iterrows():
    cid = client['client_id']
    if cid not in client_color_map:
        continue  # unassigned clients not shown
    color, emp_name, stop_num = client_color_map[cid]
    folium.CircleMarker(
        location=[client['lat'], client['lon']],
        radius=7,
        color='white',
        weight=1.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(
            f"<b>Cliënt {cid}</b><br>"
            f"Medewerker: {emp_name}<br>"
            f"Stop #{stop_num}",
            max_width=180
        ),
        tooltip=f"Cliënt {cid} | {emp_name} | stop {stop_num}"
    ).add_to(m)

print('Map built successfully.')


Map built successfully.


## 9. Legenda toevoegen en kaart opslaan

In [9]:
legend_rows = ''
for emp_idx, emp in employees_df.iterrows():
    color = EMPLOYEE_COLORS[emp_idx % len(EMPLOYEE_COLORS)]
    t     = tours[emp_idx]['total_time']
    cids  = tours[emp_idx]['client_ids']
    legend_rows += (
        '<tr>'
        f'<td><div style="width:14px;height:14px;background:{color};'
        'border:1px solid #ccc;border-radius:3px;"></div></td>'
        f'<td style="padding:0 6px;white-space:nowrap;">{emp["name"]}</td>'
        f'<td style="color:#555;white-space:nowrap;">{t:.0f} min | '
        f'cliënten: {cids}</td>'
        '</tr>'
    )

legend_html = (
    '<div style="position:fixed;bottom:20px;left:20px;z-index:1000;'
    'background:white;padding:10px 14px;border-radius:7px;'
    'font-size:11px;font-family:sans-serif;'
    'box-shadow:0 2px 10px rgba(0,0,0,.3);'
    'max-height:460px;overflow-y:auto;">'
    '<b style="font-size:13px;">Routeoverzicht</b>'
    '<table style="border-collapse:collapse;margin-top:6px;line-height:1.6;">'
    '<tr><th style="text-align:left;padding-right:6px;">Kleur</th>'
    '<th style="text-align:left;">Medewerker</th>'
    '<th style="text-align:left;padding-left:6px;">Totale tijd &amp; stops</th></tr>'
    + legend_rows +
    '</table>'
    '<hr style="margin:8px 0;">'  
    '<span style="color:#555;">● = cliënt &nbsp;&nbsp; &#9679; = thuisadres (medewerker)</span>'
    '</div>'
)

m.get_root().html.add_child(folium.Element(legend_html))

output_path = '../output/employee_routes_map.html'
m.save(output_path)
print(f'Map saved: {output_path}')

from IPython.display import IFrame, display
display(IFrame(src='employee_routes_map.html', width='100%', height=680))

Map saved: ../output/employee_routes_map.html


## 10. Samenvatting per medewerker

In [10]:
print('=== Routesamenvatting ===')
rows = []
for emp_idx, emp in employees_df.iterrows():
    t = tours[emp_idx]
    rows.append({
        'Medewerker':     emp['name'],
        'Cliënt-IDs':     str(t['client_ids']),
        'Totale tijd (min)': round(t['total_time'], 1),
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f'\nGemiddelde totale reistijd: {summary["Totale tijd (min)"].mean():.1f} min')

=== Routesamenvatting ===
  Medewerker           Cliënt-IDs  Totale tijd (min)
 employees 1  [32, 36, 76, 21, 7]                1.1
 employees 2  [73, 0, 82, 42, 35]                3.0
 employees 3 [68, 31, 83, 64, 61]                7.8
 employees 4 [38, 78, 25, 59, 37]                1.9
 employees 5 [14, 63, 93, 16, 43]                7.8
 employees 6 [28, 74, 13, 50, 12]               10.7
 employees 7 [90, 58, 97, 45, 87]               10.4
 employees 8 [72, 17, 89, 85, 39]                9.2
 employees 9 [71, 24, 22, 15, 65]               12.0
employees 10 [23, 92, 62, 69, 91]                9.4
employees 11 [51, 94, 95, 47, 29]               14.9
employees 12  [34, 9, 60, 18, 66]               10.6
employees 13   [81, 49, 88, 6, 3]               16.1
employees 14  [54, 46, 1, 77, 56]               20.1
employees 15 [19, 99, 55, 30, 79]               10.5
employees 16 [27, 96, 48, 86, 70]               15.5
employees 17  [41, 80, 2, 10, 11]               10.9
employees 18   [98, 